# Satellite Data Analysis for Jamaica
## Notebook 1 of 4. Find Your Community from Space

**Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and Technology, University of the West Indies.**

> **STUDENT EDITION.** Cells marked **YOUR TURN** have gaps to fill in. Look
> for `____` and `# TODO`. Many gaps list three options in a comment; one is
> right and the others teach you something by being wrong. Everything else runs
> as given. If you get stuck, read the hint under the cell before asking.
> The day: two hours of missions, then the one-hour Satellite Challenge.

---

# Mission 1: Count the Clear Views of Your Community 🛰️

**Your question.** Can we find our own communities from space, and how often
does the satellite actually get a clear look at them?

**Why it matters.** You cannot watch over a farm, a beach, or a disaster you
cannot reliably see. Every investigation in this course starts the same way: by
checking that the eye in the sky is really watching your patch of ground.

**Your objective.** By the end of this notebook you will measure exactly how
many clear looks one satellite got at Treasure Beach, St Elizabeth, across 2025.
First person to call out the right number wins the round.

You are now a **citizen scientist**. That means three things: you have a real
question, you go to real data, and you report an honest answer. Nobody hands you
the result. You go and get it.

### The plan for the day

| Clock | What we do |
|---|---|
| 0:00 | Notebook 1: find your community from orbit and count the clear views |
| 0:35 | Notebook 2: turn pictures into hectares |
| 1:00 | Notebook 3: measure Hurricane Melissa's damage |
| 1:30 | Notebook 4: read 45 years of Kingston heat |
| 1:50 | Breather, and mission scores |
| 2:00 | The Satellite Challenge: your team picks one investigation and presents its answer |

---

## Part 1. Set up the lab

Two cells. Press play on the first, then read on while it works. It takes about
a minute.

In [ ]:
# Run this once. On Google Colab it takes about a minute.
# If a package is already there, pip will say so and move on.
!pip install -q rasterio requests imageio pandas scikit-learn matplotlib pillow

print("Packages ready.")

In [ ]:
# 🚚 JUST RUN THIS CELL. Nothing to change. It is the toolbox for the whole course.
# ============================================================================
#  JAMAICA EARTH OBSERVATION TOOLKIT
#  Run this cell in every session. It sets up the connection to the satellite
#  archive and defines the handful of functions the whole course uses.
# ============================================================================
import os, math, json, time, warnings
warnings.filterwarnings("ignore")

# GDAL reads the satellite files straight off Amazon's servers over the
# internet. These settings tell it how to behave: no login needed, do not list
# whole directories, retry if the network hiccups.
os.environ.update({
    "AWS_NO_SIGN_REQUEST": "YES",
    "GDAL_HTTP_UNSAFESSL": "YES",
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "5",
    "GDAL_HTTP_RETRY_DELAY": "2",
})

import requests, numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from rasterio.warp import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.vrt import WarpedVRT
from PIL import Image, ImageDraw

STAC_URL = "https://earth-search.aws.element84.com/v1/search"

def _stac_post(url, body, timeout=60, tries=4):
    """POST to the archive, retrying politely if the server is having a moment."""
    for attempt in range(tries):
        try:
            r = requests.post(url, json=body, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))    # 2 s, 4 s, 6 s between tries


# House style for every chart in this course.
CYAN, INK, SAND = "#00b8d4", "#12232e", "#e0a458"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 13,
    "axes.titleweight": "bold", "figure.facecolor": "white",
})

# Places in Jamaica used through the course, as [west, south, east, north].
PLACES = {
    "kingston":     [-76.86, 17.93, -76.80, 18.02],
    "black_river":  [-77.90, 17.96, -77.78, 18.08],
    "negril":       [-78.375, 18.25, -78.320, 18.36],
    "montego_bay":  [-77.97, 18.44, -77.88, 18.51],
    "new_hope":     [-78.20, 18.13, -78.08, 18.24],
    "st_elizabeth": [-77.75, 18.00, -77.65, 18.10],
    "portland":     [-76.45, 18.10, -76.32, 18.20],
    "jamaica":      [-78.45, 17.66, -76.15, 18.55],
}

def search_scenes(bbox, start, end, max_cloud=30, limit=50, sort_by="eo:cloud_cover",
                  min_cloud=None, descending=False):
    """Ask the archive which Sentinel-2 pictures exist over a box and a date range.

    Returns a list of STAC 'items'. Each item is a dictionary of metadata plus
    links to the actual image files. Nothing is downloaded yet.

    Set `min_cloud` when you deliberately want a cloudy scene, which is useful
    for testing that your cloud masking actually works.
    """
    cloud_filter = {"lt": max_cloud}
    if min_cloud is not None:
        cloud_filter["gt"] = min_cloud
    query = {
        "collections": ["sentinel-2-l2a"],
        "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": cloud_filter},
        "limit": limit,
        "sortby": [{"field": f"properties.{sort_by}",
                    "direction": "desc" if descending else "asc"}],
    }
    r = _stac_post(STAC_URL, query, timeout=60)
    return r.json()["features"]

def search_all(bbox, start, end, max_cloud=100, page_size=100, max_pages=20):
    """Every matching scene, not just the first page.

    The archive hands back at most 200 results per request and does not warn you
    that it stopped. This follows the 'next' link until the results run out.
    """
    body = {
        "collections": ["sentinel-2-l2a"], "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": {"lt": max_cloud}},
        "limit": page_size,
        "sortby": [{"field": "properties.datetime", "direction": "asc"}],
    }
    items, url, pages, matched = [], STAC_URL, 0, None
    while url and pages < max_pages:
        r = _stac_post(url, body, timeout=90)
        j = r.json()
        items += j.get("features", [])
        matched = j.get("context", {}).get("matched", matched)
        nxt = [l for l in j.get("links", []) if l.get("rel") == "next"]
        pages += 1
        if not nxt:
            break
        url = nxt[0]["href"]
        body = nxt[0].get("body", body)
    if matched and len(items) < matched:
        print(f"Warning: got {len(items)} of {matched}. Raise max_pages.")
    return items

def make_grid(bbox, metres=20):
    """Define a fixed grid of pixels over a box, in plain latitude and longitude.

    Every image we read gets warped onto this same grid. That is what lets us
    subtract a November picture from an October one pixel by pixel, even when
    the two came from different satellite tiles in different map projections.
    """
    lon0, lat0, lon1, lat1 = bbox
    shrink = math.cos(math.radians((lat0 + lat1) / 2))
    width  = int(round((lon1 - lon0) * 111320 * shrink / metres))
    height = int(round((lat1 - lat0) * 110540 / metres))
    transform = transform_from_bounds(lon0, lat0, lon1, lat1, width, height)
    return {"width": width, "height": height, "transform": transform,
            "metres": metres, "bbox": bbox,
            "pixel_hectares": (metres * metres) / 10000.0}

def read_band(item, band, grid, resampling=Resampling.bilinear):
    """Read one colour band of one scene onto our grid. Returns raw integers."""
    with rasterio.open(item["assets"][band]["href"]) as src:
        with WarpedVRT(src, crs="EPSG:4326", transform=grid["transform"],
                       width=grid["width"], height=grid["height"],
                       resampling=resampling) as vrt:
            return vrt.read(1)

def read_reflectance(item, band, grid):
    """Read a band and convert to reflectance (0 to 1). Divide by 10000."""
    return read_band(item, band, grid).astype("float32") / 10000.0

# Scene Classification Layer codes that mean 'this pixel is usable'.
# 4 vegetation, 5 bare soil, 6 water, 7 low-probability cloud, 11 snow/ice.
CLEAR_CODES = [4, 5, 6, 7, 11]

def clear_mask(item, grid):
    """True where the pixel is usable, False where it is cloud, shadow or edge."""
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return np.isin(scl, CLEAR_CODES)

def check_coverage(item, grid):
    """How much of OUR area this scene actually covers, and how much is clear.

    The cloud percentage in the metadata describes the whole 110 km tile. It
    says nothing about your study area. Always check your own box.
    """
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return {"covered": float((scl > 0).mean()),
            "clear": float(np.isin(scl, CLEAR_CODES).mean())}

def best_scene(items, grid, min_covered=0.95, min_clear=0.60, check_n=8):
    """Walk down the candidate list and return the first scene that is genuinely
    good over our box, not just good on paper."""
    for item in items[:check_n]:
        try:
            c = check_coverage(item, grid)
        except Exception:
            continue
        if c["covered"] >= min_covered and c["clear"] >= min_clear:
            item["_coverage"] = c
            return item
    return None

def composite(items, grid, bands, max_scenes=12, min_clear=0.10, verbose=True):
    """Stack several cloud-masked scenes and take the middle value per pixel.

    One picture of Jamaica almost always has cloud somewhere. Stack ten and take
    the median and the clouds disappear, because cloud is bright and rare while
    the ground underneath is consistent.
    """
    stacks = {b: [] for b in bands}
    used = []
    for item in items:
        if len(used) >= max_scenes:
            break
        try:
            clear = clear_mask(item, grid)
            if clear.mean() < min_clear:
                continue
            for b in bands:
                a = read_reflectance(item, b, grid)
                a[~clear] = np.nan
                a[a <= 0] = np.nan
                stacks[b].append(a)
            used.append(item["properties"]["datetime"][:10])
        except Exception:
            continue
    if not used:
        raise RuntimeError("No usable scenes found. Widen the dates or raise max_cloud.")
    if verbose:
        print(f"Composite built from {len(used)} scenes: {', '.join(sorted(used))}")
    out = {b: np.nanmedian(np.stack(v), axis=0) for b, v in stacks.items()}
    out["_dates"] = sorted(used)
    return out

def normalized_difference(a, b):
    """(a - b) / (a + b). The workhorse formula behind every index in this course."""
    return (a - b) / (a + b + 1e-10)

def stretch(rgb, low=2, high=98):
    """Rescale each colour channel so the picture is bright enough to look at."""
    out = np.zeros_like(rgb, dtype="float32")
    for i in range(rgb.shape[2]):
        band = rgb[:, :, i]
        p1, p2 = np.nanpercentile(band, [low, high])
        out[:, :, i] = np.clip((band - p1) / (p2 - p1 + 1e-9), 0, 1)
    return np.nan_to_num(out)

def show(image, title="", cmap=None, vmin=None, vmax=None, bar=False, size=(9, 8)):
    """Draw an array on screen with sensible defaults."""
    fig, ax = plt.subplots(figsize=size)
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if bar:
        fig.colorbar(im, ax=ax, shrink=0.75)
    plt.tight_layout(); plt.show()

def area_hectares(mask, grid):
    """Convert a True/False mask into hectares on the ground."""
    return float(np.nansum(mask)) * grid["pixel_hectares"]

def label_frame(image_uint8, text):
    """Stamp a label bar onto one animation frame, so every frame says what it is."""
    img = Image.fromarray(image_uint8)
    draw = ImageDraw.Draw(img)
    bar = min(14 + 8 * len(text), img.width)
    draw.rectangle([0, 0, bar, 24], fill=(0, 0, 0))
    draw.text((7, 6), text, fill=(255, 255, 255))
    return np.array(img)

def save_gif(frames, path, ms=900):
    """Write labelled frames out as an animated GIF that loops forever."""
    import imageio.v2 as imageio
    imageio.mimsave(path, frames, duration=ms, loop=0)
    print(f"Saved {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {len(frames)} frames)")

def show_gif(path):
    """Play a GIF inside the notebook."""
    try:
        from IPython.display import Image as _Gif, display
        display(_Gif(filename=path))
    except Exception:
        print("Open the file from the folder panel on the left to watch it.")

print("Toolkit loaded. Study areas available:", ", ".join(PLACES))

---

## Part 2. A demo first: one year of Kingston heat, animated 🎞️

Before you understand a single line, watch what you are about to be able to do.

The cell below asks NASA (the National Aeronautics and Space Administration) for a whole year of temperature over Kingston, then
paints each month by how hot it was and plays them as a calendar. Cool months
sit blue. Hot months burn red. Press play and watch the year heat up and cool
down.

You will build every part of this yourself over the next two hours.

In [ ]:
# 🚚 JUST RUN THIS. Your first taste of what satellite and climate data can do.
import io, requests
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib import colors

# one year of monthly temperature for Kingston, straight from NASA POWER
resp = requests.get("https://power.larc.nasa.gov/api/temporal/monthly/point",
                    params=dict(parameters="T2M", community="AG",
                                latitude=17.98, longitude=-76.80,
                                start=2025, end=2025, format="JSON"), timeout=60)
temp = resp.json()["properties"]["parameter"]["T2M"]

names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
vals  = [temp[f"2025{m:02d}"] for m in range(1, 13)]     # 12 numbers, one per month

heat = plt.get_cmap("turbo")                             # blue = cool, red = hot
scale = colors.Normalize(min(vals) - 0.5, max(vals) + 0.5)

def draw_year(upto):
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, (nm, v) in enumerate(zip(names, vals)):
        row, col = divmod(i, 4)
        face = heat(scale(v)) if i <= upto else (0.9, 0.9, 0.9, 1)   # grey until revealed
        ax.add_patch(plt.Rectangle((col, 2 - row), 0.92, 0.92, color=face))
        if i == upto:                                                # highlight this month
            ax.add_patch(plt.Rectangle((col, 2 - row), 0.92, 0.92,
                                       fill=False, lw=4, edgecolor="black"))
        ax.text(col + 0.46, 2 - row + 0.46, nm, ha="center", va="center", fontweight="bold",
                color="white" if i <= upto else "black")
    ax.text(2, 3.4, f"KINGSTON  {vals[upto]:.1f} C", ha="center", fontsize=20, fontweight="bold")
    ax.text(2, -0.45, f"{names[upto]} 2025", ha="center", fontsize=13)
    ax.set_xlim(-0.1, 4.1); ax.set_ylim(-0.8, 3.8); ax.axis("off")
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    buf.seek(0); return imageio.imread(buf)

frames = [draw_year(i) for i in range(12)] + [draw_year(11)] * 3     # hold on the last frame
os.makedirs("outputs", exist_ok=True)
save_gif(frames, "outputs/kingston_year.gif", ms=450)
show_gif("outputs/kingston_year.gif")

---

## Part 3. Python basics: variables, lists, loops, decisions

That took about twenty lines of Python. Here is everything you need to read
lines like those. Six short cells. Run each one, read what it prints, change a
number, run it again. You cannot break these.

In [ ]:
# 1. A variable is a name for a value. The equals sign stores it.
parish = "St Elizabeth"        # text in quotes is a string
population = 152_074           # underscores just make big numbers readable

print("Parish:", parish)
print("People:", population)

In [ ]:
# 2. Maths works as you expect. This line converts Celsius to Fahrenheit.
temp_c = 31.5
temp_f = temp_c * 9 / 5 + 32

# an f in front of the quotes drops values into text at the {curly braces}
print(f"{temp_c} C is {temp_f:.1f} F")     # :.1f shows one decimal place

In [ ]:
# 3. A list holds several values in a row. Counting starts at 0, not 1.
parishes = ["Kingston", "St Andrew", "Portland", "St Mary"]

print("First:", parishes[0])       # position 0
print("Last :", parishes[-1])      # -1 means last
print("Count:", len(parishes))     # len counts the items

# 4. A loop repeats. The indented line runs once for each item.
for p in parishes:
    print("  Hello,", p)

In [ ]:
# 5. if and else make a decision. This is exactly how we pick clear pictures.
cloud_cover = 12

if cloud_cover < 20:
    print("Clear enough. Use it.")
else:
    print("Too cloudy. Skip it.")

In [ ]:
# 6. A function is a named recipe. Write once, use as often as you like.
def km_tall(lat_bottom, lat_top):
    return (lat_top - lat_bottom) * 110.54     # one degree of latitude is 110.54 km

print(f"{km_tall(17.93, 18.02):.1f} km")       # the Kingston box, later in Part 5

### The computer draws the picture, you supply the meaning

Run the next cell. Look at what appears. Shout out what animal it is.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

t = np.linspace(0, 2 * np.pi, 200)          # points around a circle
fig, ax = plt.subplots(figsize=(6, 5))
ax.axhspan(-2.3, -0.9, color="#8fd8f7")                                 # water to float on
ax.fill(2.2 * np.cos(t), 1.3 * np.sin(t), color="#FFD43B")             # body
ax.fill([-1.9, -3.1, -1.6], [0.2, 1.3, 0.85], color="#FFD43B")        # tail
ax.fill(0.95 * np.cos(t) + 1.55, 0.95 * np.sin(t) + 1.55, color="#FFD43B")   # head
ax.fill([2.35, 3.45, 2.4], [1.75, 1.5, 1.3], color="#FF8C00")         # beak
ax.plot(1.82, 1.82, "o", color="black", ms=8)                          # eye
ax.set_xlim(-3.6, 3.8); ax.set_ylim(-2.3, 3.1); ax.axis("off")
ax.set_title("MYSTERY ANIMAL", fontweight="bold")
plt.show()

The computer drew every line. It has no idea it made a duck. **You** supplied
that. Hold onto this, because it is the whole job: the machine does the sums,
the scientist says what they mean. It comes back in Notebook 4, when a program
sorts Kingston into groups and cannot name a single one.

---

## Part 4. Four ideas: bands, reflectance, cloud cover, coordinates

Four plain ideas carry this whole course. None needs maths.

### Idea 1: a satellite takes many pictures at once

Your phone takes one picture. It mixes red, green and blue into the colours you
see, and stops there.

A Sentinel-2 satellite takes **thirteen pictures at the same instant**, each in
a different slice of light. Some slices are colours your eye can see. Some,
like infrared, are invisible to you but not to the sensor. Think of thirteen
cameras stacked together, each behind a different filter, all pressing the
shutter together. Each slice is called a **band**, and later you will mix bands
the way you mix paint.

### Idea 2: reflectance is how much light a surface bounces back

Point a torch at a surface. Some light is swallowed, some bounces back. The
fraction that bounces back is the **reflectance**, a number from 0 to 1. Zero
means it swallowed everything, like a black hole. One means it bounced back
everything, like a perfect mirror.

You already know this in your body. A black shirt in the Kingston sun bakes
because it swallows light. A white shirt stays cooler because it bounces light
away. The satellite measures exactly that fraction, for every pixel, in every
band. Run this to see typical values.

In [ ]:
import matplotlib.pyplot as plt

surface = ["Clear water", "Asphalt road", "Green grass", "Bare soil", "Dry sand", "Fresh snow"]
bounces = [0.04,          0.06,           0.15,          0.20,        0.35,       0.80]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(surface, bounces, color=plt.get_cmap("YlOrBr")([b for b in bounces]))
ax.set_xlim(0, 1); ax.set_xlabel("Reflectance   (0 = swallows all light,  1 = mirror)")
ax.set_title("How much light everyday surfaces bounce back")
for y, b in enumerate(bounces):
    ax.text(b + 0.02, y, f"{b:.2f}", va="center")
plt.tight_layout(); plt.show()

# Cloud tops bounce back about 0.9, near the top of this chart.
# That is exactly why cloud ruins a picture: it is brighter than anything below it.

### Idea 3: cloud cover is a percentage

A percentage is a count out of 100. If a picture is 25% cloud, then a quarter of
it is hidden and three quarters is clear. Zero percent is a perfectly clear
picture. One hundred percent is a total white-out with no ground in sight.

When you search the archive you set a limit, say 20%, and only reasonably clear
pictures come back. The grid below is one hundred squares; the shaded ones show
what 25% looks like.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 4))
for i in range(100):                       # 100 squares, a 10 by 10 grid
    row, col = divmod(i, 10)
    cloudy = i < 25                        # shade the first 25 to show 25%
    ax.add_patch(plt.Rectangle((col, row), 0.9, 0.9,
                               color="#b0b0b0" if cloudy else "#8fd8f7"))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("25% cloud: a quarter hidden, the rest clear")
plt.show()

### Idea 4: latitude and longitude pin down any place

Every place on Earth has two numbers.

**Latitude** is how far north or south you are. The Equator is 0. North is a
positive number, south is negative. Jamaica sits near 18 north.

**Longitude** is how far east or west you are. The line through Greenwich in
London is 0. East is positive, west is negative. Jamaica sits near 77 **west**,
so its longitude is written as a negative number, -77.

Run the map. The dot is Jamaica. Notice how the grid lets you name any point
with just those two numbers.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
for lat in range(16, 21):
    ax.axhline(lat, color="#dddddd")       # lines of latitude run across
for lon in range(-80, -73):
    ax.axvline(lon, color="#dddddd")       # lines of longitude run up and down

ax.plot(-77.3, 18.1, "o", ms=16, color=CYAN)
ax.annotate("JAMAICA\n18 N, 77 W", (-77.3, 18.1),
            xytext=(10, 10), textcoords="offset points", fontsize=12, fontweight="bold")
ax.annotate("north", (-79.6, 19.7), fontweight="bold")
ax.annotate("east ->", (-75.6, 16.3), fontweight="bold")
ax.set_xlabel("Longitude   (west of Greenwich is negative)")
ax.set_ylabel("Latitude   (north of the Equator is positive)")
ax.set_title("Two numbers name any place on Earth")
plt.tight_layout(); plt.show()

---

## Part 5. The bounding box: four numbers that mark your area

A computer cannot look up "Black River" by name. It needs a **bounding box**:
four numbers marking a rectangle, in this order.

```
[west, south, east, north]
```

Two longitudes and two latitudes, exactly the numbers from Idea 4. Kingston is
`[-76.86, 17.93, -76.80, 18.02]`. Get the order wrong and the archive returns
nothing at all.

In [ ]:
kingston = PLACES["kingston"]          # ready-made boxes live in the toolbox
print("Kingston box:", kingston)

west, south, east, north = kingston    # give the four numbers names
wide = (east - west) * 111.32 * math.cos(math.radians((south + north) / 2))
tall = (north - south) * 110.54
print(f"{wide:.1f} km wide, {tall:.1f} km tall")

### YOUR TURN 1

Montego Bay is near 18.47 north, 77.92 west. Build a box roughly 10 km on a
side, which means 0.045 either side of the centre.

In [ ]:
# Pick four of these six values. Two are traps that land in the sea:
#   -77.965   -77.875   77.875   18.425   18.515   17.425
mobay = [____, ____, ____, ____]       # order: west, south, east, north

west, south, east, north = mobay
wide = (east - west) * 111.32 * math.cos(math.radians((south + north) / 2))
print(f"{wide:.1f} km wide")

assert west < east < 0, "West of Greenwich is negative, and west is the smaller number."
assert south < north, "South is the smaller latitude."
print("Box looks right.")

*Hint: two of the six values break a rule from Idea 4. One longitude is
positive, which puts it on the wrong side of the planet. One latitude is too far
south, out at sea. Find the four that obey the rules.*

---

## Part 6. Search the satellite archive for your box

The archive is a giant, searchable index of every picture the satellite has
taken. You send a box, a date range, and a cloud limit. It sends back a list.
Nothing downloads yet, so this is fast and free.

In [ ]:
scenes = search_scenes(
    bbox=kingston,
    start="2025-01-01",
    end="2025-06-30",
    max_cloud=25,           # only pictures 25% cloudy or less (Idea 3)
)

print(f"{len(scenes)} pictures match.\n")
for s in scenes[:5]:
    p = s["properties"]
    print(f"  {p['datetime'][:10]}   cloud {p['eo:cloud_cover']:5.1f}%   {s['id']}")

In [ ]:
scene = scenes[0]                # the list arrives sorted clearest-first

# a scene ID packs in the satellite, the grid tile, the date and the level
print("Scene:", scene["id"])
print("Date :", scene["properties"]["datetime"][:10])
print("\nOne file per band (Idea 1):")
for band in ["blue", "green", "red", "nir", "swir16", "scl"]:
    print(f"  {band:8s} -> {scene['assets'][band]['href'].split('/')[-1]}")

Those bands are the thirteen pictures from Idea 1. The ones you will lean on:

| Band | Slice of light | Used for |
|---|---|---|
| `blue`, `green`, `red` | The colours you see | Normal pictures |
| `nir` | Near infrared (NIR), invisible to you | Living plants bounce it hard |
| `swir16` | Shortwave infrared (SWIR) | Moisture, bare ground, burnt land |
| `scl` | The scene classification layer (SCL): a label per pixel | Deciding what to throw away |

The one fact to keep: plants **swallow red** for photosynthesis and **bounce
infrared** away so they do not overheat. Notebook 2 is built on it.

### Meet Sentinel-2, the satellites behind these pictures

Look at the IDs you just printed. S2A, S2B and S2C are three spacecraft run by
the European Space Agency (ESA), free to everyone on Earth by law: 786 km up,
back over the same ground every five days, always crossing Jamaica near 10:30
in the morning so the light matches between visits. The full fact sheet is on
the intro slides.

---

## Part 7. Download the pixel values inside your box

First set a **grid**: a fixed layout of pixels over the box. Every picture we
read is snapped onto this same grid, which is what lets us line one date up
against another later.

In [ ]:
grid = make_grid(kingston, metres=20)     # each pixel is 20 m by 20 m of ground

print(f"{grid['width']} across, {grid['height']} down")
print(f"{grid['pixel_hectares']} hectares per pixel")

In [ ]:
red = read_reflectance(scene, "red", grid)     # downloads only our box, not the whole tile

print("Shape  :", red.shape)
print("Average:", round(float(np.nanmean(red)), 4))
print("\nRaw numbers from a 4 by 4 corner:")
print(np.round(red[100:104, 100:104], 3))

Those are reflectance values (Idea 2): the fraction of red light each pixel
bounced back. The two pictures below are the same data. On the left it is an
image. On the right it is a count of how many pixels hold each value.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.imshow(red, cmap="Greys_r", vmin=0, vmax=0.3)
ax1.set_title("Red band over Kingston"); ax1.axis("off")

ax2.hist(red.ravel(), bins=120, range=(0, 0.4), color=CYAN)
ax2.set_title("How many pixels hold each value")
ax2.set_xlabel("Reflectance")

plt.tight_layout(); plt.show()

# Two humps: the tall one is the sea swallowing red light, the wide one is the city.

---

## Part 8. True colour: red, green and blue make a photo

Your screen builds colour from red, green and blue. Stack those three bands in
that order and you get the view from an aeroplane window.

### YOUR TURN 2

In [ ]:
green = read_reflectance(scene, "green", grid)
blue  = read_reflectance(scene, "blue", grid)

# Fill the three slots. Options: red / green / blue, each used once.
# The name of the format tells you the order.
rgb = np.dstack([____, ____, ____])

assert rgb.shape[2] == 3, "Three channels needed."
show(stretch(rgb), "Kingston in true colour")

*Hint: the format is called RGB. Read that left to right.*

`stretch` brightens the picture for your eye. Every measurement in this course
uses the raw numbers, never the brightened ones. Looking and measuring are
different jobs.

---

## Part 9. False colour: infrared makes living plants glow

Your eye stops at red. The satellite keeps going. Put **near infrared** where
red normally sits and living plants blaze bright red, because they bounce
infrared so hard. Water turns near black. Concrete turns pale. No human eye has
ever seen this directly.

### YOUR TURN 3

In [ ]:
nir = read_reflectance(scene, "nir", grid)

# Options: nir / red / green. The invisible band takes the first slot.
false_colour = np.dstack([____, ____, ____])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6))
a1.imshow(stretch(rgb));          a1.set_title("True colour"); a1.axis("off")
a2.imshow(stretch(false_colour)); a2.set_title("False colour: red = living plants"); a2.axis("off")
plt.tight_layout(); plt.show()

*Hint: infrared is the band your eye cannot see, and it goes first.*

### The blink test: flick between two images to spot change

Astronomers used to flick between two photographs and let their eyes catch what
moved. Clyde Tombaugh found Pluto that way in 1930. A GIF (Graphics Interchange Format, a short looping animation) is the same trick.
Every frame gets a label, because an animation without one is a puzzle, not a
result.

In [ ]:
when = scene["properties"]["datetime"][:10]

a = label_frame((stretch(rgb) * 255).astype("uint8"),          f"TRUE COLOUR  {when}")
b = label_frame((stretch(false_colour) * 255).astype("uint8"), f"FALSE COLOUR  {when}")

save_gif([a, b], "outputs/kingston_blink.gif", ms=900)
show_gif("outputs/kingston_blink.gif")

---

## Part 10. The cloud percentage covers the whole tile, not your box

The cloud percentage from Part 6 describes the **whole 110 km tile**, not your
box. Jamaica sits across two satellite grid zones, so many scenes are thin
slivers: 0.0% cloud, only because there is almost nothing in them to be cloudy.

Trust that number without checking and you will analyse a rectangle of nothing.
Watch it happen over Black River, the town Melissa hit hardest.

In [ ]:
br_grid = make_grid(PLACES["black_river"], metres=20)
maybe = search_scenes(PLACES["black_river"], "2025-10-01", "2025-10-27", max_cloud=60)

print(f"{'Date':12s} {'Tile cloud':>11s} {'Covers OUR box':>15s}")
print("-" * 40)
for item in maybe[:5]:
    cov = check_coverage(item, br_grid)          # measure coverage ourselves
    print(f"{item['properties']['datetime'][:10]:12s} "
          f"{item['properties']['eo:cloud_cover']:10.1f}% "
          f"{cov['covered'] * 100:14.1f}%")

In [ ]:
# best_scene walks the list and takes the first that truly covers us
pick = best_scene(maybe, br_grid, min_covered=0.95, min_clear=0.30)

if pick:
    print("Picked:", pick["id"])
else:
    print("Nothing covers Black River properly in October 2025.")
    print("An honest 'no' beats a confident wrong answer.")

It found nothing, and it was right to. No single October scene covers Black
River properly, which is exactly why Notebook 3 stacks part-scenes together. The
rule to carry with you: **the cloud number describes what the provider cared
about, not what you care about. Always check your own box.**

In [ ]:
# The same tool succeeding somewhere easier, Negril
ne_grid = make_grid(PLACES["negril"], metres=20)
pick = best_scene(search_scenes(PLACES["negril"], "2025-01-01", "2025-06-30", max_cloud=25),
                  ne_grid, min_covered=0.95, min_clear=0.70)
print("Picked:", pick["id"], pick["properties"]["datetime"][:10])

ne_rgb = np.dstack([read_reflectance(pick, x, ne_grid) for x in ("red", "green", "blue")])
show(stretch(ne_rgb), f"Negril, {pick['properties']['datetime'][:10]}", size=(6, 9))

---

## 📍 Your spot: view the place that matters to you

Open Google Maps. Right-click the place that matters most to you: your home,
your school, your beach. The top line of the menu is the latitude and longitude.
Maps gives latitude first, so keep that order and drop them in below.

In [ ]:
MY_LAT, MY_LON = ____, ____            # for example 18.4113, -77.1367 for Dunn's River

spot = [MY_LON - 0.02, MY_LAT - 0.02, MY_LON + 0.02, MY_LAT + 0.02]    # a 4 km box
sg = make_grid(spot, metres=10)

sc = best_scene(search_scenes(spot, "2025-01-01", "2025-12-31", max_cloud=20),
                sg, min_covered=0.95, min_clear=0.7)
assert sc is not None, "No clean scene here. Try max_cloud=35, or nudge the point inland."

pic = np.dstack([read_reflectance(sc, x, sg) for x in ("red", "green", "blue")])
show(stretch(pic), f"Your spot, {sc['properties']['datetime'][:10]}", size=(7, 7))

---

## 🏁 Mission objective: count the clear views of Treasure Beach in 2025

Treasure Beach took Melissa hard. Farms flattened, boats wrecked. Before anyone
can measure any of that, one question comes first, and it is a citizen
scientist's first question every time. Is the satellite even watching this
place, and how often does it get a clear look?

**Find the value.** How many pictures under 15% cloud did Sentinel-2 capture
over Treasure Beach across 2025?

Complete the two blanks, run it, and call out your number. First correct answer
wins the round.

In [ ]:
treasure = [-77.79, 17.87, -77.75, 17.91]      # Treasure Beach, St Elizabeth

# Blank 1, the cloud limit: your mission asks for pictures under 15% cloud.
# Blank 2, the count: which function from Part 3 counts items in a list?
found = search_scenes(treasure, "2025-01-01", "2025-12-31",
                      max_cloud=____, limit=200)

print("THE VALUE:", ____(found))

*Stuck? Both answers are things you have already used. The cloud limit is a
whole number stated in your mission brief. The counting function appeared in
Part 3, cell 3, next to a list of parishes. Neither is a decimal.*

---

## Mission 1 complete

- Four numbers pin down any place: west, south, east, north
- A satellite takes thirteen pictures at once, not one
- Reflectance is the fraction of light a surface bounces back
- False colour reveals living plants your eye cannot see
- **Always check coverage over your own box**

You found your community from orbit and measured how often the satellite sees
it. In Mission 2 you stop looking and start measuring, turning these pictures
into hectares. In the final hour, the Satellite Challenge puts all of it in
your team's hands.

---

### Before you close this notebook

Save a copy to your own Drive (`File` then `Save a copy in Drive`). The next
notebook assumes you have this one working.

*Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and
Technology, University of the West Indies.*

*Satellite Data Analysis for Jamaica. Built with free, open data: Sentinel-2 from
the European Space Agency, hosted by Amazon; NASA POWER climate records. No API
keys, no fees, no permission needed.*